# Results Comparison

**Purpose**: score one run against another, overall and by tool size, thinness, scene challenge and tool-free frames.

**Input**: `per_image_results.csv` and `results.json` for the selected runs (local cache, GCS fallback).
Every run was scored on identical frames and folds, so this is a re-aggregation rather than a re-run.

**Output**: tables and figures under `outputs/<arm>_vs_<reference>/`. With `OUTPUT_ALL = False` the last section rebuilds that directory for every declared pair.


## 1. Install dependencies
Run this cell to install the packages required for this notebook.


In [ ]:
from pathlib import Path

_req = Path("requirements.txt")
if not _req.exists():
    _req = Path("02_results_comparison") / "requirements.txt"
if not _req.exists():
    raise FileNotFoundError(f"requirements.txt not found from cwd={Path.cwd()}")
get_ipython().run_line_magic("pip", f"install -q -r {_req.as_posix()}")

import matplotlib, numpy, pandas, scipy
print(f"pandas {pandas.__version__} | numpy {numpy.__version__} "
      f"| matplotlib {matplotlib.__version__} | scipy {scipy.__version__}")


## 2. Setup and run selection
Pick `REFERENCE` and `ARM` from the registry printed below. `EXCLUDE_PROBLEMATIC` drops frames marked as annotation errors (none currently). Override the keys if comparing a different pair.


In [ ]:
import sys
from pathlib import Path

_NB_DIR = Path.cwd()
if not (_NB_DIR / "compare_common.py").exists():
    _NB_DIR = _NB_DIR / "02_results_comparison"
if not (_NB_DIR / "compare_common.py").exists():
    raise ModuleNotFoundError(f"compare_common.py not found from cwd={Path.cwd()}")
sys.path.insert(0, str(_NB_DIR.resolve()))

import importlib
import compare_common
importlib.reload(compare_common)  # pick up edits to compare_common without a kernel restart
from compare_common import *
from IPython.display import Image, display

# ── Selection ──
REFERENCE = "baseline-seed2026"           # the run everything below is measured against
ARM = "cascade-redetect-clean"            # the run measured against it
EXCLUDE_PROBLEMATIC = False               # future work: no frame carries the status yet
OUTPUT_ALL = False                        # section 11: rebuild outputs/ for every declared pair

display(registry())


### 2a. Load the pair
Load both runs, assert they share frames, folds and ground truth, then set the output directory.


In [ ]:
ARMS = [ARM]
frames, gt = load_runs([REFERENCE] + ARMS, exclude_problematic=EXCLUDE_PROBLEMATIC)
strata = build_strata(gt)

OUT = output_dir(REFERENCE, ARMS, EXCLUDE_PROBLEMATIC)
TABLES, FIGURES = OUT / "tables", OUT / "figures"

print(f"reference : {REFERENCE:<28s} {results(REFERENCE)['run_id']}  {RUNS[REFERENCE].label}")
print(f"arm       : {ARM:<28s} {results(ARM)['run_id']}  {RUNS[ARM].label}")
print(f"frames    : {len(gt):,}  ({int(strata['all tool frames'].sum()):,} tool-present, "
      f"{int(strata['tool-free frames'].sum()):,} tool-free)")
print(f"excluded  : {gt.attrs['excluded']} annotation errors"
      + ("" if EXCLUDE_PROBLEMATIC else "  (EXCLUDE_PROBLEMATIC is off)"))
print(f"strata    : size cuts {' / '.join(f'{e:g}' for e in SIZE_BAND_EDGES)} % of frame area,"
      f" thin/wide cut {thin_cut(REFERENCE):.4f} px at 512 x 512")
print("grids     : " + ", ".join(f"{k} {len(stored_thresholds(frames[k]))} points"
                                 for k in [REFERENCE] + ARMS))
print(f"output    : {OUT}")
for note in warnings_for(REFERENCE, ARMS):
    print(f"\n{note}")


## 3. Headline numbers
Fold mean and standard deviation from each run's `results.json`.


In [ ]:
display(save_table(headline([REFERENCE] + ARMS), "headline_fold_means", TABLES)[
    ["label", "run_id", "iou", "iou_sd", "dice", "precision", "recall",
     "iou_tiny", "iou_small", "iou_med", "iou_large", "iou_thin", "iou_wide",
     "notool_clean_rate", "hd95", "msd"]])


## 4. Where the reference fails
IoU, error per tool pixel, and share of the IoU deficit by size and thinness. False-positive mass on tool-present vs tool-free frames.


In [ ]:
diagnostics = error_distribution(frames, strata, REFERENCE)
print(f"Failure distribution on {RUNS[REFERENCE].label}, tool-present frames")
display(save_table(diagnostics["by_stratum"], "reference_error_by_stratum", TABLES))
print("Where the false-positive mass sits")
display(save_table(diagnostics["fp_location"], "reference_fp_location", TABLES))

display(Image(filename=str(figure_error_distribution(diagnostics["by_stratum"], REFERENCE, FIGURES))))


## 5. Paired per-frame differences
Arm minus reference on the same frames. The interval resamples whole videos; the test is Wilcoxon signed-rank.


In [ ]:
paired = save_table(paired_deltas(frames, gt, strata, REFERENCE, ARMS), "paired_deltas", TABLES)
print("Paired per-frame IoU")
display(paired[paired["metric"] == "iou"][
    ["stratum", "label", "n_frames", "n_videos", "reference", "arm_value", "delta", "median",
     "ci_low", "ci_high", "p", "improved_pct", "verdict"]])

display(Image(filename=str(figure_forest(paired, REFERENCE, ARMS, FIGURES))))
display(Image(filename=str(figure_stratum_bars(frames, strata, REFERENCE, ARMS, FIGURES))))


## 6. Matched precision
Recall at the threshold where each arm's precision equals the reference's, beside the same comparison at threshold 0.50.

1. **per-frame** - mean over frames, the convention used everywhere else.
2. **pooled** - confusion counts summed over the stratum first, which weights large tools more.


In [ ]:
matched = save_table(matched_precision(frames, strata, REFERENCE, ARMS), "matched_precision", TABLES)
print("Recall at the reference's own precision")
display(matched[["convention", "stratum", "label", "n_frames", "target_precision",
                 "matched_threshold", "delta_recall_matched", "delta_recall_at_0.50",
                 "out_of_grid"]])

sweep = save_table(threshold_sweep(frames, strata, REFERENCE, ARMS), "threshold_sweep", TABLES)
display(Image(filename=str(figure_operating_curves(sweep, matched, REFERENCE, ARMS, FIGURES))))


### 6a. Per-frame distribution
The matched-precision mean broken down per frame, so a few large wins can be told from a broad small change.


In [ ]:
display(save_table(matched_distribution(frames, strata, REFERENCE, ARMS),
                   "matched_distribution", TABLES))

for stratum in DISTRIBUTION_STRATA:
    display(Image(filename=str(
        figure_delta_distribution(frames, strata, REFERENCE, ARMS, stratum, FIGURES))))


## 7. Threshold recalibration
Threshold picked on four folds and applied to the fifth, rotating. Reported for the reference as well.


In [ ]:
display(save_table(lofo_threshold(frames, gt, strata, REFERENCE, ARMS), "threshold_lofo", TABLES))


## 8. Tool-free frames
False positives on frames with no instrument. `clean_rate` is the share with fewer than 100 false-positive pixels.


In [ ]:
display(save_table(tool_free(frames, strata, REFERENCE, ARMS), "tool_free", TABLES))


## 9. Cascade mechanism
Which cascade tier fired and what it cost. Skip if the selected arm is not a cascade. `02b` opens this up properly.


In [ ]:
if all("cascade_tier" in frames[a] for a in ARMS):
    for arm in ARMS:
        diag = cascade_diagnostics(frames, strata, arm, REFERENCE)
        print(f"{RUNS[arm].label}, against {RUNS[REFERENCE].label}")
        display(save_table(diag["by_tier"], f"cascade_tiers_{arm}", TABLES))
        display(save_table(diag["by_zoom"], f"cascade_zoom_{arm}", TABLES))
else:
    print("no cascade run selected")


## 10. Every run at a glance
Every scored run from its own `results.json`, sorted by IoU.


In [ ]:
every_run = master_table().sort_values("iou", ascending=False)
every_run["erroneous"] = every_run["erroneous"].ne("").map({True: "yes", False: ""})
display(every_run[["label", "family", "varies", "run_id", "iou", "iou_sd", "iou_tiny",
                   "iou_small", "iou_med", "iou_large", "iou_thin", "iou_wide",
                   "notool_clean_rate", "erroneous"]].round(4))


## 11. Regenerate every comparison
With `OUTPUT_ALL = False`, rebuild `outputs/` for every pair in the registry. Overwrites its own outputs only.


In [ ]:
if OUTPUT_ALL:
    made = build_all_outputs(cleaned=EXCLUDE_PROBLEMATIC)
    print(f"\n{len(made)} directories under {OUTPUT_ROOT}")
else:
    print(f"OUTPUT_ALL is off; only {OUT.name} was written")
